In [1]:
import re
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
MAX_WORDS = 6000
CHAPTER_SPLIT = re.compile(r"\n{4,}")
TEXT_SEPARATORS = ["\n\n", "\n", ". ", "! ", "? "]

In [3]:


def split_into_chapters(text):
    """
    Split text into chapters using 4+ newlines as a delimiter.
    """
    # Normalize Windows-style newlines
    text = text.replace("\r\n", "\n")
    
    # Split on 4 or more consecutive newlines
    parts = [p.strip() for p in CHAPTER_SPLIT.split(text) if p.strip()]
    n_chapters = len(parts)
    chapters = {"Chapter " + str(i): parts[i] for i in range(n_chapters)}
    return chapters


def split_chapters(chapter_content: str, max_words: int):
    """
    Split chapter content into smaller chunks with a maximum number of words.
    """
    # RecursiveCharacterTextSplitter lets you split hierarchically
    splitter = RecursiveCharacterTextSplitter(
        separators=TEXT_SEPARATORS,  # hierarchy of separators
        chunk_size=max_words,       # maximum words per chunk
        chunk_overlap=0,        # no overlap
        length_function=lambda x: len(x.split())  # count words instead of chars
    )
    chunks = splitter.split_text(chapter_content)
    return chunks


def create_chunks(book: str):
    """
    Create list of chunks from the book text.
    Each chunk is a dictionary with 'Chapter', 'Part', and 'Content'.
    """
    chapters = split_into_chapters(book)
    chunks = []

    for chapter, content in chapters.items():
        splitted = split_chapters(content, MAX_WORDS)
        chunks += [
            {
                "Chapter": chapter,
                "Part": "Part " + str(i),
                "Content": splitted[i],
                }
            for i in range(len(splitted))]
    return chunks

In [4]:
# Example of use

with open('../../data/books/Dracula - Bram Stoker.txt', 'r', encoding='utf-8') as file:
    book = file.read()

chunks = create_chunks(book)

# Save to a JSON file
with open("../../data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks[:10], f, indent=4, ensure_ascii=False)

In [5]:
for chunk in chunks[:20]:
    print(len(chunk['Content'].split()), "words in", chunk['Chapter'], chunk['Part'])

9 words in Chapter 0 Part 0
44 words in Chapter 1 Part 0
5 words in Chapter 2 Part 0
144 words in Chapter 3 Part 0
77 words in Chapter 4 Part 0
1 words in Chapter 5 Part 0
5700 words in Chapter 6 Part 0
5487 words in Chapter 7 Part 0
5728 words in Chapter 8 Part 0
5894 words in Chapter 9 Part 0
3546 words in Chapter 10 Part 0
5715 words in Chapter 11 Part 0
5677 words in Chapter 12 Part 0
5873 words in Chapter 13 Part 0
446 words in Chapter 13 Part 1
5951 words in Chapter 14 Part 0
5942 words in Chapter 15 Part 0
5127 words in Chapter 16 Part 0
5667 words in Chapter 17 Part 0
1624 words in Chapter 17 Part 1
